# UAV Pathfinding + LoRA fine-tuning of a ~1B LLM -- Colab quickstart

Fine-tunes a small open LLM (default: `meta-llama/Llama-3.2-1B-Instruct`) with LoRA to solve a simple UAV grid pathfinding task (fly from `S` to `G` avoiding obstacles `#`), and scores it against the BFS-optimal path.

**Before running:** Runtime -> Change runtime type -> GPU (T4 is fine, that's what the default config is tuned for).

If you're not opening this notebook from inside an already-cloned repo, edit `REPO_URL` in the next cell first.

In [ ]:
REPO_URL = "https://github.com/<your-username>/<your-repo>.git"  # <-- edit this
REPO_DIR = REPO_URL.rstrip('/').split('/')[-1].replace('.git', '')

import os
if not os.path.isdir(REPO_DIR):
    !git clone $REPO_URL
%cd $REPO_DIR

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q -r requirements.txt

## (Only if using a gated model like Llama-3.2-1B-Instruct)

1. Accept the license at https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct ("Agree and access repository").
2. Create a token at https://huggingface.co/settings/tokens (read access is enough).
3. Run the cell below and paste the token when prompted.

Skip this if you switched `configs/default.yaml` -> `model.name` to an ungated model such as `Qwen/Qwen2.5-0.5B-Instruct` or `HuggingFaceTB/SmolLM2-1.7B-Instruct`.

In [ ]:
from huggingface_hub import login
login()

## 1. Generate the dataset (BFS ground truth)

In [ ]:
!python -m src.data_gen --config configs/default.yaml

## 2. Evaluate the base model zero-shot (before fine-tuning) -- baseline for comparison

`--limit 50` keeps this quick; drop it to evaluate the full test set.

In [ ]:
!python -m src.evaluate --config configs/default.yaml \
    --report_path outputs/eval_report_base.json --limit 50

## 3. LoRA fine-tune

Edit `configs/default.yaml` first if you want to change the model, grid sizes, dataset size, batch size, etc. Defaults are sized for a free T4 (16GB) Colab GPU.

In [ ]:
!python -m src.train --config configs/default.yaml

## 4. Evaluate the fine-tuned model

In [ ]:
!python -m src.evaluate --config configs/default.yaml \
    --adapter_dir outputs/lora-pathfinding \
    --report_path outputs/eval_report_finetuned.json

In [ ]:
import json
base = json.load(open("outputs/eval_report_base.json"))["summary"]
ft = json.load(open("outputs/eval_report_finetuned.json"))["summary"]
print("BASE      :", base)
print("FINE-TUNED:", ft)

## 5. Visualize one example (BFS-optimal path vs. the model's actual path)

In [ ]:
!python -m src.visualize --config configs/default.yaml --split test --index 0 \
    --eval_report outputs/eval_report_finetuned.json --out outputs/example.png

from IPython.display import Image
Image("outputs/example.png")

## Notes

- Metrics reported (see `README.md` for details): `success_rate`, `avg_length_ratio_on_success` (1.0 = as short as the BFS-optimal path), `invalid_move_rate`, `unparseable_rate`.
- If you hit an out-of-memory error on T4, lower `train.per_device_train_batch_size` in `configs/default.yaml` and raise `train.gradient_accumulation_steps` to compensate, or lower `data.grid_size_range` / `model.max_length`.
- Before touching the real model, you can sanity-check the whole pipeline offline in seconds/minutes with `configs/smoke_test.yaml` (see `scripts/smoke_test.sh`) -- it uses a tiny randomly-initialized model and a char-level tokenizer, no GPU or internet needed.